# scANVI

In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
import pycats
from pandas.api.types import CategoricalDtype

scvi.settings.seed = 1234
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
adata = sc.read("Reproducibility/Data/External_cohort/adata_concat_w_external_cohort.h5ad")

In [ ]:
adata_TNK = adata[adata.obs['predicted_lineage'].isin(['TNK']) & adata.obs['coarse_celltype'].isin(['unknown', 'CD4_Tconv', 'CD8_T', 'Treg', 'NK_ILC'])
                  & adata.obs['FACS'].isin(['bulk','CD45pos'])].copy()

In [ ]:
def scANVI_integration(adata,tmp_lineage):
    sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            flavor='seurat_v3',
                            layer="counts",
                            batch_key="paper",
                            subset=True)
    ###
    adata_DOGMA = adata[adata.obs['paper'].isin(['DOGMA'])==True].copy()
    adata_EXT = adata[adata.obs['paper'].isin(['DOGMA'])==False].copy()
    ###
    ## Integration with scVI ##
    scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="batch_id")
    scvi_model = scvi.model.SCVI(adata, n_layers=2, n_latent=30)
    ###
    scvi_model.train(max_epochs=1000, early_stopping=True)
    ###
    subset_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/'
    os.makedirs(subset_path, exist_ok = True)
    scvi_model_path = f'{subset_path}integration_batch_id/'
    scvi_model.save(scvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    ax.legend()
    plt.savefig(f'{scvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    SCVI_LATENT_KEY = "X_scVI"
    adata.obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()
    ###
    sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["paper",'batch_id','celltype'],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{scvi_model_path}Atlas_level_integration_UMAP.pdf', bbox_inches='tight')
    plt.close()

In [ ]:
def scANVI_annotation_transfer(adata,tmp_lineage,celltype_use,categories_order):
    subset_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/'
    scvi_model_path = f'Reproducibility/Results/scANVI/BC/{tmp_lineage}/integration_batch_id/'
    scvi_model = scvi.model.SCVI.load(scvi_model_path, adata)
    ## Transfer of annotations with scANVI ##
    SCANVI_CELLTYPE_KEY = celltype_use
    scanvi_model = scvi.model.SCANVI.from_scvi_model(
        scvi_model,
        adata=adata,
        unlabeled_category="unknown",
        labels_key=SCANVI_CELLTYPE_KEY,
    )
    ###
    scanvi_model.train(max_epochs=1000, early_stopping=True, 
                       n_samples_per_label=adata.obs[SCANVI_CELLTYPE_KEY].value_counts().min())
    ###
    scanvi_model_path = f'{subset_path}SCANVI_{SCANVI_CELLTYPE_KEY}/'
    scanvi_model.save(scanvi_model_path, overwrite=True)
    ###
    fig, ax = plt.subplots(1, 1)
    scanvi_model.history["elbo_train"].plot(ax=ax, label="train")
    scanvi_model.history["elbo_validation"].plot(ax=ax, label="validation")
    ax.set(title="Negative ELBO over training epochs")
    plt.savefig(f'{scanvi_model_path}ELBO_curve_plot.pdf', bbox_inches='tight')
    plt.close()
    ###
    ## Load
    # scanvi_model = scvi.model.SCANVI.load(scanvi_model_path, adata)
    ## scanvae = sca.models.SCANVI.load(scanvi_model, adata)
    ###
    SCANVI_LATENT_KEY = "X_scANVI"
    SCANVI_PREDICTION_KEY = f'predicted_{SCANVI_CELLTYPE_KEY}'
    ###
    adata.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata)
    adata.obs[SCANVI_PREDICTION_KEY] = scanvi_model.predict(adata)
    ###
    sc.pp.neighbors(adata, use_rep=SCANVI_LATENT_KEY)
    sc.tl.umap(adata, min_dist=0.5)
    ###
    categories_order2 = list(filter(lambda x: x != 'unknown', categories_order))
    ###
    adata.obs[SCANVI_CELLTYPE_KEY] = pd.Categorical(adata.obs[SCANVI_CELLTYPE_KEY], categories=categories_order, ordered=True)
    adata.obs[SCANVI_PREDICTION_KEY] = pd.Categorical(adata.obs[SCANVI_PREDICTION_KEY], categories=categories_order2, ordered=True)
    ###
    sc.set_figure_params(figsize=(3, 3))
    sc.set_figure_params(vector_friendly=True, dpi_save=100)
    sc.pl.umap(
        adata,
        color=["paper",'batch_id','celltype',SCANVI_PREDICTION_KEY],
        frameon=False,
        ncols=4,
    #    size=0.3,
        wspace = 0.5,
    #    legend_loc="on data",
        legend_fontsize=4,
        show = False
    )
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_UMAP.pdf', bbox_inches='tight')
    plt.close()
    ###
    # concordance
    tmp = adata[adata.obs['paper'].isin(['DOGMA'])==True].copy()
    ###
    accuracy = np.mean(tmp.obs[SCANVI_PREDICTION_KEY].astype(str) == tmp.obs[SCANVI_CELLTYPE_KEY].astype(str))
    print("Acc: {}".format(accuracy))
    ###
    df = tmp.obs.groupby([SCANVI_CELLTYPE_KEY, SCANVI_PREDICTION_KEY]).size().unstack(fill_value=0)
    norm_df = df / df.sum(axis=0)
    ###
    plt.figure(figsize=(8, 8))
    _ = plt.pcolor(norm_df)
    _ = plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
    _ = plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
    plt.xlabel("Predicted")
    plt.ylabel("Observed")
    plt.text(0.5, 1.05, f'Acc: {accuracy:.4f}', fontsize=12, ha='center', transform=plt.gca().transAxes)
    plt.savefig(f'{subset_path}Transfer_annotation_SCANVI_concordance_corrplot.pdf', bbox_inches='tight')
    plt.close()
    ###
    # save
    adata.obs.to_csv(f'{subset_path}Atlas_level_integration_{tmp_lineage}_metadata.txt', sep='\t')

In [ ]:
categories_order_TNK = ["CD4_Tn","CD4_Tcm",'CD4_Tsen',"CD4_T_CD26","CD4_CTL","CD4_Th17","CD4_Tfh-like",
                        "CD4_Tph-like","CD4_T_proliferative","Treg_naive","Treg_effector",
                        "CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1",
                        "CD8_Tex_2","CD8_T_proliferative","NK_CD56_CD49a_Hi_CD103_Hi","NK_CD56_CD49a_Hi_CD103_Lo",
                        "NK_CD56_CD49a_Lo","NK_CD56_dim",'ILC3','MAIT', 'unknown']

In [ ]:
scANVI_integration(adata = adata_TNK, tmp_lineage = 'TNK')
scANVI_annotation_transfer(adata = adata_TNK, 
                           tmp_lineage = 'TNK', 
                           celltype_use = 'celltype', 
                           categories_order=categories_order_TNK)

## scVI with Pan-cancer CD4<sup>+</sup> T

In [ ]:
adata = sc.read("Reproducibility/Data/External_cohort/adata_concat_w_pancancer_CD4_T.h5ad")
adata = adata[adata.obs['cancerType'].isin(['CHOL'])==False]

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            flavor='seurat_v3',
                            layer="counts",
                            subset=True)

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="libraryID")

model = scvi.model.SCVI(adata)
model.view_anndata_setup()

In [ ]:
model.train(max_epochs = 1000, early_stopping=True)
model.save("path_to_save", overwrite = True)